# 00 · Build the unified dataset

**Shared pipeline notebook (team-owned).** Run it once after `tools/get_data.py`, and again whenever this notebook or the raw data changes. Changes go through a PR.

**Outputs** (both gitignored, so each person builds their own copy):
- `data/clean/films_unified.csv`: **one row per film** with every source attached, keyed by `film_uid`. Includes per-film review summaries (`rtrev_*`, `slrev_*`).
- `data/clean/reviews_unified.csv`: **one row per critic review** (one film has many), keyed by `film_uid` (§9b).

**`film_uid`** is the stable key: `tmdb-<TMDB id>`, or `tn-<The Numbers URL slug>` for films not in TMDB. It doesn't change between rebuilds. All other notebooks start from this file instead of re-merging.

| Source | Adds | Join |
|---|---|---|
| TMDB | Base table: one row per TMDB id | — |
| The Numbers | Budget, **worldwide** gross, MPAA, source, franchise, companies | Title + year matcher (§4, hand-checked 50/50) plus a looser second pass (§4b) |
| Wikidata snapshot (committed) | Budget and box office in USD | TMDB id |
| Box Office Mojo | Domestic gross, theaters (US theatrical run) | Title + year matcher |
| RT stefanoleone992 | `actors`, `production_company` | Title + year matcher |
| RT andrezaza | Director, writer, RT box office, ratings | Matt's join rules (§7), hand-checked 0/50 wrong |

**Combined financials:** `fin_budget` / `fin_revenue` take the first available value from TMDB → The Numbers → Wikidata; `fin_*_src` records which. `usable` = 2000–2023, budget ≥ $1M, revenue > 0.

Rows not in TMDB (The Numbers only) have a **negative `id`** (not stable across rebuilds; use `film_uid`) and `in_tmdb` = False.

Column prefixes: none = TMDB, `tn_` The Numbers, `wd_` Wikidata, `bom_` Box Office Mojo, `sl_` RT stefanoleone992, `rt_` RT andrezaza, `fin_` combined.

## 1. Setup

In [ ]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
YEARS = (2000, 2023)          # contract window
BUDGET_FLOOR = 1_000_000      # contract floor

## 2. TMDB

All released, non-adult films (one row per id), plus films the snapshot still lists as in production but whose release date had passed (late 2023). Matching uses a wider year window (1998–2025) so films near the edges can still match; the contract window is applied after.

In [ ]:
t = pd.read_csv(RAW / "TMDB_movie_dataset_v11.csv", low_memory=False)
t["release_dt"] = pd.to_datetime(t["release_date"], errors="coerce")
t["year"] = t["release_dt"].dt.year
# Released films, plus films the snapshot still lists as in production but whose release date is before
# its 2023 cutoff (e.g. Wonka, Wish, Five Nights at Freddy's, Eras Tour). Without them, their The Numbers
# figures matched same-title junk entries instead (found 2026-11-24, Edwards-A10 §8e).
late = t["status"].isin(["Post Production", "In Production"]) & t["release_dt"].le("2023-12-31")
tmdb = t[(t["status"].eq("Released") | late) & ~t["adult"] & t["year"].between(1998, 2025)].drop_duplicates("id").copy()
print(f"Added {int((late & ~t['adult'] & t['year'].between(1998, 2025)).sum()):,} late-2023 films listed as still in production")

# Sanity check: Matt's contract population is reproduced exactly
pop = tmdb[tmdb["year"].between(*YEARS) & tmdb["budget"].ge(BUDGET_FLOOR) & tmdb["revenue"].gt(0)]
assert len(pop) == 5766, len(pop)
print(f"TMDB candidates for matching: {len(tmdb):,}; contract population: {len(pop):,}")

## 3. Load the other sources

In [ ]:
tn = pd.read_csv(RAW / "the_numbers/Top Movies (Cleaned Data).csv", low_memory=False)
tn["year"] = pd.to_datetime(tn["Release Date"], errors="coerce").dt.year
tn = tn[tn["year"].between(*YEARS)].reset_index(drop=True)

wd = pd.read_csv(RAW / "wikidata/wikidata_film_financials_2026-11-21.csv")
wd = wd.rename(columns={"tmdb": "id"}).drop_duplicates("id")

bom_daily = pd.read_csv(RAW / "boxofficemojo/revenues_per_day.csv.gz", parse_dates=["date"])
# `id` in this file is per ROW (film-day), not per film. Collapse to one row per theatrical RUN: the same title
# can be different films (Frozen 2010 vs Frozen 2013), so a gap of more than 180 days between grossing days
# starts a new run. Grouping by title alone summed different films' grosses (found 2026-11-26, Edwards §16d).
bom_daily = bom_daily.sort_values(["title", "date"])
bom_daily["run"] = bom_daily.groupby("title")["date"].diff().dt.days.gt(180).groupby(bom_daily["title"]).cumsum()
bom = (bom_daily.groupby(["title", "run"])
       .agg(first_date=("date", "min"), domestic_gross=("revenue", "sum"),
            days=("date", "nunique"), max_theaters=("theaters", "max"), distributor=("distributor", "first"))
       .reset_index().drop(columns="run"))
bom["year"] = bom["first_date"].dt.year

sl = pd.read_csv(RAW / "stefanoleone992/rotten_tomatoes_movies.csv", low_memory=False)
sl["year"] = pd.to_datetime(sl["original_release_date"], errors="coerce").dt.year

pd.Series({"The Numbers (2000-23)": len(tn), "Wikidata": len(wd), "Box Office Mojo titles": len(bom),
           "RT stefanoleone992": len(sl)}, name="rows").to_frame()

## 4. Title + year matcher

1. **Normalize titles:** strip accents, lowercase, `&` → `and`, drop a leading *The / A / An*, keep letters and digits only.
2. **Candidates:** same normalized title as TMDB's `title` **or** `original_title`, release year within ±2.
3. **Pick the best candidate** per source film: budget agrees (±15%), then closest year, then closest runtime, then most TMDB votes (votes only break ties for identity; they're never a model input).
4. **Year differences need corroboration:** if years differ, runtime must be within 10 minutes or the budget must agree.
5. **One-to-one only:** a TMDB film claimed by two source films is dropped from both.

In [ ]:
def norm_title(s: pd.Series) -> pd.Series:
    s = s.fillna("").astype(str).map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode())
    s = s.str.lower().str.replace("&", " and ", regex=False)
    s = s.str.replace(r"^(the|a|an)\s+", "", regex=True)
    return s.str.replace(r"[^a-z0-9]", "", regex=True)


tmdb_keys = pd.concat([
    tmdb.assign(key=norm_title(tmdb["title"])),
    tmdb.assign(key=norm_title(tmdb["original_title"])),
]).drop_duplicates(["id", "key"])
tmdb_keys = tmdb_keys.loc[tmdb_keys["key"].ne(""), ["id", "key", "year", "runtime", "budget", "vote_count"]]


def match_to_tmdb(src, title_col, runtime_col=None, budget_col=None, name="src", keys=None):
    """Return src_row -> film id, one-to-one, with match diagnostics. `keys` defaults to all TMDB films."""
    keys = tmdb_keys if keys is None else keys
    s = pd.DataFrame({"src_row": np.arange(len(src)), "key": norm_title(src[title_col]), "src_year": src["year"]})
    s["src_runtime"] = src[runtime_col] if runtime_col else np.nan
    s["src_budget"] = src[budget_col] if budget_col else np.nan
    m = s.merge(keys, on="key")
    m["year_diff"] = (m["src_year"] - m["year"]).abs()
    m = m[m["year_diff"] <= 2]
    m["runtime_diff"] = (m["src_runtime"] - m["runtime"]).abs()
    m["budget_agrees"] = m["budget"].gt(0) & ((m["budget"] / m["src_budget"] - 1).abs() < 0.15)
    m = (m.sort_values(["src_row", "budget_agrees", "year_diff", "runtime_diff", "vote_count"],
                       ascending=[True, False, True, True, False], na_position="last")
          .drop_duplicates("src_row"))
    m = m[m["year_diff"].eq(0) | m["runtime_diff"].le(10) | m["budget_agrees"]]
    m = m[~m["id"].duplicated(keep=False)]
    print(f"{name:<12} matched {len(m):,} of {len(src):,} ({len(m) / len(src):.0%}); "
          f"year diff counts {m['year_diff'].value_counts().sort_index().to_dict()}")
    return m[["src_row", "id", "year_diff", "runtime_diff", "budget_agrees"]]


tn_match = match_to_tmdb(tn, "Movie Name", "Running Time (minutes)", "Production Budget (USD)", "The Numbers")
bom_match = match_to_tmdb(bom, "title", name="Box Office Mojo")
sl_match = match_to_tmdb(sl, "movie_title", "runtime", name="RT sl")

both = tn_match.merge(tmdb[["id", "budget"]], on="id")
both = both[both["budget"].gt(0)]
print(f"\nCheck: where TMDB and The Numbers both have a budget, they agree (±15%) for "
      f"{both['budget_agrees'].mean():.0%} of {len(both):,} matched films")

## 4b. Second pass for The Numbers: looser titles

The Numbers writes titles its own way, so §4's exact-title rule missed about 120 TMDB films, mostly big ones: *Star Wars **Ep. VII**: The Force Awakens*, *Ocean's **8***, ***Dr. Seuss'** The Lorax*, *Kill Bill: **Volume** 1*, truncated titles ending in "…". Each became a duplicate "The Numbers-only" row.

This pass matches the leftovers to TMDB films not already claimed:

1. **Loose title key:** drops parentheticals, brand prefixes (*Disney's*, *Dr. Seuss'*, *Tyler Perry's*), "3D", filler words (the, part, episode, movie); Roman numerals and number words become digits. Similarity score 0–1; a truncated or subtitle-dropped title that fits inside the other counts as a match.
2. **Year within ±1** and **money agrees**, either:
   - **A:** budget within 15%, score ≥ 0.85, runtime within 10 min (5 if the years differ), or
   - **B:** worldwide revenue within 10%, score ≥ 0.80, runtime within 30 min. Needed because budgets often disagree (*The Force Awakens*: $245M in TMDB, $533M in The Numbers).
3. **One-to-one**, best score first.

Not caught (titles too different, e.g. *X-Men 2* vs *X2*, *F9: The Fast Saga* vs *F9*): about 10 films stay as separate rows. Some leftovers are genuinely missing from the TMDB file (*The Marvels*, *Aquaman and the Lost Kingdom*).

In [ ]:
import difflib, re

ROMAN = {"i": "1", "ii": "2", "iii": "3", "iv": "4", "v": "5", "vi": "6", "vii": "7", "viii": "8", "ix": "9"}
WORDS = {"two": "2", "three": "3", "four": "4", "eight": "8", "sixty": "60"}
FILLER = {"the", "a", "an", "part", "episode", "movie"}


def loose_title(x) -> str:
    """Looser key for The Numbers' title style: 'Star Wars Ep. III', 'Ocean's 8', 'Dr. Seuss' The Grinch', truncated '…'."""
    x = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode().lower()
    x = x.replace("&", " and ").replace("...", " ")
    x = re.sub(r"\(.*?\)|\[.*?\]", " ", x)                      # parentheticals: foreign titles, '(2016)'
    x = re.sub(r"^(disney|dr\.? seuss|tyler perry|marvel)\S*\s+", "", x)   # possessive brand prefixes
    x = re.sub(r"\b3-?d\b|\bimax\b", " ", x)
    x = re.sub(r"\bep\b\.?", "episode", x)
    toks = [ROMAN.get(t, WORDS.get(t, t)) for t in re.sub(r"[^a-z0-9 ]", " ", x).split()]
    return " ".join(t for t in toks if t not in FILLER)


def title_score(a_raw, b_raw):
    a, b = loose_title(a_raw), loose_title(b_raw)
    s = difflib.SequenceMatcher(None, a, b).ratio()
    if "…" in str(a_raw) and b.startswith(a):                  # The Numbers truncates long titles
        s = max(s, 0.99)
    if min(len(a), len(b)) >= 6 and (a in b or b in a):        # subtitle added/dropped
        s = max(s, 0.9)
    return s


# Second pass for The Numbers films §4 left unmatched, against TMDB films not already claimed.
# A candidate needs release year within ±1, plus money that agrees, plus a similar title:
#   A) budget within 15%, loose-title score >= 0.85, runtime within 10 min or unknown (a year gap needs runtime within 5), or
#   B) worldwide revenue within 10%, loose-title score >= 0.80, runtime within 30 min or unknown
#      (budgets often disagree: e.g. Star Wars: The Force Awakens is $245M in TMDB vs $533M in The Numbers)
un = tn.drop(index=tn_match["src_row"])
# TMDB side limited to 2000-2023: a match outside the window would drop the film from the table
# (e.g. Godzilla Minus One 2023 -> its 2024 black-and-white re-release)
free = tmdb.loc[~tmdb["id"].isin(tn_match["id"]) & tmdb["year"].between(*YEARS) & (tmdb["budget"].gt(0) | tmdb["revenue"].gt(0)),
                ["id", "title", "original_title", "year", "runtime", "budget", "revenue"]]
cands = []
for i, r in un.iterrows():
    b, g = r["Production Budget (USD)"], r["Worldwide Gross (USD)"]
    c = free[(free["year"] - r["year"]).abs().le(1)]
    bud_ok = (c["budget"] > 0) & ((c["budget"] / b - 1).abs() < 0.15) if b > 0 else pd.Series(False, index=c.index)
    rev_ok = (c["revenue"] > 0) & ((c["revenue"] / g - 1).abs() < 0.10) if g > 0 else pd.Series(False, index=c.index)
    for j, s in c[bud_ok | rev_ok].iterrows():
        cands.append({"src_row": i, "id": s["id"], "year_diff": abs(s["year"] - r["year"]),
                      "runtime_diff": abs(s["runtime"] - r["Running Time (minutes)"]) if s["runtime"] > 0 else np.nan,
                      "budget_agrees": bool(bud_ok[j]), "revenue_agrees": bool(rev_ok[j]),
                      "score": max(title_score(r["Movie Name"], s["title"]), title_score(r["Movie Name"], s["original_title"]))})
c2 = pd.DataFrame(cands)
rd = c2["runtime_diff"]
rule_a = c2["budget_agrees"] & c2["score"].ge(0.85) & (rd.le(10) | rd.isna()) & (c2["year_diff"].eq(0) | rd.le(5))
rule_b = c2["revenue_agrees"] & c2["score"].ge(0.80) & (rd.le(30) | rd.isna())
c2 = c2[rule_a | rule_b]
c2 = (c2.sort_values(["src_row", "score", "year_diff", "runtime_diff"], ascending=[True, False, True, True], na_position="last")
        .drop_duplicates("src_row"))
c2 = c2[~c2["id"].duplicated(keep=False)]
tn_match = pd.concat([tn_match, c2[tn_match.columns]], ignore_index=True)
assert tn_match["src_row"].is_unique and tn_match["id"].is_unique
show = c2.merge(tmdb[["id", "title"]], on="id").assign(tn_title=lambda d: tn.loc[d["src_row"], "Movie Name"].values)
print(f"Second pass matched {len(c2)} more The Numbers films (total {len(tn_match):,} of {len(tn):,}; "
      f"{c2['budget_agrees'].sum()} by budget, {(~c2['budget_agrees']).sum()} by revenue only)")
show.sort_values("score")[["tn_title", "title", "year_diff", "runtime_diff", "budget_agrees", "score"]].head(15)


## 5. Attach the sources to TMDB

In [ ]:
def attach(base, src, match, prefix, cols):
    part = src.iloc[match["src_row"]][cols].set_axis(match["id"].values)
    part.columns = [f"{prefix}_{c}" for c in cols]
    part[f"{prefix}_year_diff"] = match["year_diff"].values
    return base.merge(part, left_on="id", right_index=True, how="left")


# Every column from each source (except its own id / derived year)
tn_cols = [c for c in tn.columns if c not in ("id", "year")]
bom_cols = [c for c in bom.columns if c != "year"]
sl_cols = [c for c in sl.columns if c != "year"]
films = tmdb.copy()
films = attach(films, tn, tn_match, "tn", tn_cols)
films = attach(films, bom, bom_match, "bom", bom_cols)
films = attach(films, sl, sl_match, "sl", sl_cols)
films = films.merge(wd.add_prefix("wd_").rename(columns={"wd_id": "id"}), on="id", how="left")
assert films["id"].is_unique
print(f"{films.shape[1]} columns after attaching The Numbers ({len(tn_cols)}), Box Office Mojo ({len(bom_cols)}), "
      f"RT stefanoleone992 ({len(sl_cols)}), Wikidata ({wd.shape[1] - 1})")

## 6. Combined budget and revenue

First non-missing value wins: TMDB → The Numbers → Wikidata. Zero means missing in every source.

In [ ]:
def first_valid(*cols):
    vals = pd.concat([c.where(c > 0) for c, _ in cols], axis=1)
    src = pd.Series(pd.NA, index=vals.index, dtype="string")
    for i, (_, label) in reversed(list(enumerate(cols))):
        src = src.mask(vals.iloc[:, i].notna(), label)
    return vals.bfill(axis=1).iloc[:, 0], src


films["fin_budget"], films["fin_budget_src"] = first_valid(
    (films["budget"], "tmdb"), (films["tn_Production Budget (USD)"], "the_numbers"), (films["wd_wd_budget_usd"], "wikidata"))
films["fin_revenue"], films["fin_revenue_src"] = first_valid(
    (films["revenue"], "tmdb"), (films["tn_Worldwide Gross (USD)"], "the_numbers"), (films["wd_wd_box_office_usd"], "wikidata"))

films["usable"] = (films["year"].between(*YEARS) & films["fin_budget"].ge(BUDGET_FLOOR) & films["fin_revenue"].gt(0))
usable = films[films["usable"]]
print(f"Usable films: {len(usable):,} (TMDB alone: {len(pop):,}, +{len(usable) - len(pop):,})")
pd.crosstab(usable["fin_budget_src"], usable["fin_revenue_src"], margins=True)

## 6c. Add The Numbers films that have no TMDB match

The Numbers films (2000–2023) that matched no TMDB film in §4 or §4b. They're added as their own rows so the table covers every film we have financials for. They have The Numbers fields (budget, gross, MPAA, genre, companies…) but no TMDB fields.

- **Duplicate guard:** a film is *not* added if TMDB already has a film with the same normalized title within ±1 year (and any financial value). The matcher in §4 rejected those pairs only because nothing corroborated the year difference (e.g. *Jason X*: 2002 vs 2001, budgets $14M vs $11M); they're the same film, and keeping both would count it twice. A second guard in §7 drops any that share an RT record with a TMDB film (catches bigger year gaps, e.g. a 2001 Hong Kong release vs a 2004 US release).
- **`id` is negative** for these rows (−1, −2, …): there's no TMDB id. `in_tmdb` = False marks them.
- `title`, `year`, `release_dt` and `runtime` are filled from The Numbers so the RT join (§7) can match them too.
- `fin_*` come from The Numbers; `usable` uses the same rule as every other film.

In [ ]:
tn_new = tn.drop(index=tn_match["src_row"])

# Duplicate guard: same normalized title as a TMDB film with financials, year within +/-1
has_fin = films["year"].between(*YEARS) & (films["fin_budget"].notna() | films["fin_revenue"].notna())
tk = films.loc[has_fin, ["title", "original_title", "year"]]
tmdb_titles = pd.concat([pd.DataFrame({"key": norm_title(tk[c]), "y": tk["year"]}) for c in ["title", "original_title"]])
probe = pd.DataFrame({"key": norm_title(tn_new["Movie Name"]), "y": tn_new["year"]}, index=tn_new.index)
hits = probe.reset_index().merge(tmdb_titles, on="key", suffixes=("", "_t"))
dup_idx = hits.loc[(hits["y"] - hits["y_t"]).abs() <= 1, "index"].unique()
print(f"Not added (probable duplicates of TMDB films): {len(dup_idx)}")
tn_new = tn_new.drop(index=dup_idx).reset_index(drop=True)
rows = tn_new[tn_cols].add_prefix("tn_")
rows.insert(0, "id", -(np.arange(len(tn_new)) + 1))
rows["title"] = tn_new["Movie Name"].values
rows["original_title"] = tn_new["Movie Name"].values
rows["release_dt"] = pd.to_datetime(tn_new["Release Date"], errors="coerce").values
rows["year"] = tn_new["year"].values
rows["runtime"] = tn_new["Running Time (minutes)"].values
rows["tn_year_diff"] = 0
rows["fin_budget"] = rows["tn_Production Budget (USD)"].where(rows["tn_Production Budget (USD)"] > 0)
rows["fin_revenue"] = rows["tn_Worldwide Gross (USD)"].where(rows["tn_Worldwide Gross (USD)"] > 0)
rows["fin_budget_src"] = rows["fin_budget"].notna().map({True: "the_numbers", False: pd.NA})
rows["fin_revenue_src"] = rows["fin_revenue"].notna().map({True: "the_numbers", False: pd.NA})
rows["usable"] = rows["year"].between(*YEARS) & rows["fin_budget"].ge(BUDGET_FLOOR) & rows["fin_revenue"].gt(0)
rows["in_tmdb"] = False

films["in_tmdb"] = True
before = films["usable"].sum()
films = pd.concat([films, rows], ignore_index=True)
assert films["id"].is_unique
print(f"Added {len(rows):,} The Numbers-only films ({rows['usable'].sum():,} usable). "
      f"Usable films: {before:,} -> {films['usable'].sum():,}")

## 6d. Match Box Office Mojo and RT stefanoleone992 to The Numbers-only films

§4 matched these two sources to TMDB films only. The Numbers-only films are matched here with the same matcher, against source records that weren't already used (keeps every source record on at most one film).

In [ ]:
new = films[~films["in_tmdb"]]
new_keys = pd.DataFrame({"id": new["id"], "key": norm_title(new["title"]), "year": new["year"],
                         "runtime": new["runtime"], "budget": new["fin_budget"], "vote_count": 0})


def fill_from(src, used_rows, title_col, runtime_col, prefix, cols, name):
    global films
    pool = src.drop(index=used_rows).reset_index(drop=True)
    m = match_to_tmdb(pool, title_col, runtime_col, name=name, keys=new_keys)
    part = pool.iloc[m["src_row"]][cols].set_axis(m["id"].values)
    part.columns = [f"{prefix}_{c}" for c in cols]
    part[f"{prefix}_year_diff"] = m["year_diff"].values
    idx = films.index[films["id"].isin(part.index)]
    films.loc[idx, part.columns] = part.loc[films.loc[idx, "id"]].values


fill_from(bom, bom_match["src_row"], "title", None, "bom", bom_cols, "BOM -> TN-only")
fill_from(sl, sl_match["src_row"], "movie_title", "runtime", "sl", sl_cols, "RT sl -> TN-only")

## 7. RT movies (andrezaza): director, writer, RT box office

Uses **Matt's join rules exactly** (`Sorensen-A10.ipynb` §4, hand-checked 0/50 wrong): normalized title (`title`, then `original_title`), RT year within ±1, or rule 2b (streaming-only RT date later than release, runtime within 5 min); best candidate = title before original_title, rule 2 before 2b, closest year, closest runtime; one-to-one, ambiguous pairs dropped.

Matt ran it on the 5,766-film population. Here it runs on every 2000–2023 film with any financial value, after first checking that the same code reproduces his 4,673 pairs.

In [ ]:
def matt_norm(s):
    s = s.fillna("").astype(str).map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode())
    s = s.str.lower().str.replace("&", " and ", regex=False)
    s = s.str.replace(r"[^a-z0-9 ]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def parse_money(s):
    m = s.str.extract(r"^\$([\d.]+)([KMB]?)$")
    return m[0].astype(float) * m[1].map({"": 1, "K": 1e3, "M": 1e6, "B": 1e9})


rt_movies = pd.read_csv(RAW / "rotten_tomatoes_movies.csv", low_memory=False)
rtj = rt_movies.drop_duplicates("id").copy()
rt_theat = pd.to_datetime(rtj["releaseDateTheaters"], errors="coerce").dt.year
rt_stream = pd.to_datetime(rtj["releaseDateStreaming"], errors="coerce").dt.year
rtj["rt_year"] = rt_theat.fillna(rt_stream)
rtj["rt_year_source"] = np.where(rt_theat.notna(), "theatrical", np.where(rt_stream.notna(), "streaming", "none"))
rtj["key"] = matt_norm(rtj["title"])
rtj = rtj[rtj["key"].ne("") & rtj["rt_year"].notna()]


def rt_join(tj):
    """Matt's §4 rules. tj needs id, title, original_title, year, runtime. Returns the 1:1 crosswalk."""
    tj = tj.assign(key_title=matt_norm(tj["title"]), key_orig=matt_norm(tj["original_title"]))
    side = rtj[["id", "key", "rt_year", "rt_year_source", "runtimeMinutes"]].rename(columns={"id": "rt_id"})
    cands = []
    for key_col, how in [("key_title", "title"), ("key_orig", "original_title")]:
        c = tj[["id", "year", "runtime", key_col]].merge(side, left_on=key_col, right_on="key")
        cands.append(c.assign(matched_on=how))
    cands = pd.concat(cands, ignore_index=True)
    signed = cands["rt_year"] - cands["year"]
    cands["year_diff"] = signed.abs()
    cands["runtime_diff"] = (cands["runtimeMinutes"] - cands["runtime"].replace(0, np.nan)).abs()
    rule2 = cands["year_diff"] <= 1
    rule2b = (cands["rt_year_source"] == "streaming") & (signed > 1) & (cands["runtime_diff"] <= 5)
    cands["rule"] = np.where(rule2, "2: year +/-1", np.where(rule2b, "2b: later streaming + runtime", None))
    cands = cands[rule2 | rule2b].copy()
    cands["title_rank"] = cands["matched_on"].map({"title": 0, "original_title": 1})
    cands["rule_rank"] = cands["rule"].str.startswith("2b").astype(int)
    cands = cands.sort_values(["id", "title_rank", "rule_rank", "year_diff", "runtime_diff"], na_position="last")
    best = cands.drop_duplicates("id")
    return best[~best["rt_id"].duplicated(keep=False)][["id", "rt_id", "matched_on", "rule", "year_diff", "runtime_diff"]]


# 1) Same code on Matt's population must give his 4,673 pairs
assert len(rt_join(pop)) == 4673, "doesn't reproduce Matt's crosswalk"

# 2) Three passes, so verified films get first claim on each RT record:
#    usable TMDB films -> usable The Numbers-only films -> all other 2000-2023 films
scope = films["year"].between(*YEARS) & (films["fin_budget"].notna() | films["fin_revenue"].notna())
passes = [films["usable"] & films["in_tmdb"], films["usable"] & ~films["in_tmdb"], scope & ~films["usable"]]
taken, parts = set(), []
for mask in passes:
    xw = rt_join(films[mask])
    xw = xw[~xw["rt_id"].isin(taken)]
    taken |= set(xw["rt_id"])
    parts.append(xw)
rt_xw = pd.concat(parts, ignore_index=True)
assert rt_xw["id"].is_unique and rt_xw["rt_id"].is_unique

# The Numbers-only films whose RT match was already claimed by a TMDB film: probably the same film
tn_side = rt_join(films[passes[1]])
dup_suspects = tn_side[tn_side["rt_id"].isin(parts[0]["rt_id"])]

matt_pairs = rt_join(pop)
kept = parts[0].merge(matt_pairs, on=["id", "rt_id"]).shape[0]
print(f"RT matches: {len(parts[0]):,} usable TMDB + {len(parts[1]):,} usable The Numbers-only + {len(parts[2]):,} other. "
      f"Matt's verified pairs kept: {kept:,} of {len(matt_pairs):,}")
# Second duplicate guard: sharing an RT record with a TMDB film means it's the same film
# (e.g. Shaolin Soccer: 2001 Hong Kong release vs 2004 US release in The Numbers)
films = films[~films["id"].isin(dup_suspects["id"])]
print(f"Dropped {len(dup_suspects)} The Numbers-only films that share an RT record with a TMDB film (duplicates)")

In [ ]:
rt_cols = [c for c in rt_movies.columns if c != "id"]
rt_part = (rt_xw.merge(rtj[["id"] + rt_cols].rename(columns={"id": "rt_id"}), on="rt_id")
               .assign(boxOffice_usd=lambda d: parse_money(d["boxOffice"])))
rt_part = rt_part.set_index("id").add_prefix("rt_").rename(columns={"rt_rt_id": "rt_id"})
films = films.drop(columns=[c for c in films.columns if c.startswith("rt_")]).merge(
    rt_part, left_on="id", right_index=True, how="left")
assert films["id"].is_unique
films.loc[films["usable"], ["rt_id", "rt_director", "rt_writer", "rt_boxOffice_usd"]].notna().mean().map("{:.0%}".format)

## 7b. Critic reviews, rolled up to one row per film

Two review files, each keyed by an RT slug, so they attach through the RT links already on each film (no title matching):

| File | Attaches via | Prefix |
|---|---|---|
| `rotten_tomatoes_movie_reviews.csv` (1.44M) | `rt_id` (RT andrezaza join) | `rtrev_` |
| `stefanoleone992/rotten_tomatoes_critic_reviews.csv` (1.13M) | `sl_rotten_tomatoes_link` | `slrev_` |

Only **copies of the same review** are dropped: repeated `reviewId` in the andrezaza file (each repeat is identical in every column), and rows identical in every column (including the review text) in the stefanoleone992 file. Different reviews of the same film are all kept; the full list is saved separately in §9b. Per film: review count, top-critic count, % fresh, first and last review date, and how many reviews are dated **before** the TMDB release date (plan step 11, "early buzz").

⚠️ Reviews are known only after release, except the before-release counts. **They're descriptive columns, not model inputs** (contract: no post-release inputs). Review text is not copied (license: short excerpts only).

In [ ]:
def rollup(rev, key, fresh, top, date, prefix, release):
    """One row per film key: counts, % fresh, date range, reviews dated before release."""
    r = pd.DataFrame({"key": rev[key], "fresh": fresh, "top": top.astype(bool),
                      "date": pd.to_datetime(rev[date], errors="coerce")})
    r = r.merge(release, on="key", how="inner")
    r["before"] = r["date"] < r["release_dt"]
    g = r.groupby("key").agg(n_reviews=("fresh", "size"), n_top_critic=("top", "sum"),
                             pct_fresh=("fresh", "mean"), first_review=("date", "min"),
                             last_review=("date", "max"), n_before_release=("before", "sum"))
    g["pct_fresh"] = (100 * g["pct_fresh"]).round(1)
    return g.add_prefix(prefix)


# RT andrezaza reviews -> rt_id
rev_a = pd.read_csv(RAW / "rotten_tomatoes_movie_reviews.csv", low_memory=False,
                    usecols=["id", "reviewId", "creationDate", "criticName", "publicatioName", "isTopCritic", "originalScore", "reviewState"])
n0 = len(rev_a)
rev_a = rev_a.drop_duplicates("reviewId")          # Matt §2c: 24,788 rows share a reviewId; every one is identical in all columns
print(f"RT andrezaza reviews: dropped {n0 - len(rev_a):,} repeated reviewIds")
rel_a = films.loc[films["rt_id"].notna(), ["rt_id", "release_dt"]].rename(columns={"rt_id": "key"})
agg_a = rollup(rev_a, "id", rev_a["reviewState"].eq("fresh"), rev_a["isTopCritic"], "creationDate", "rtrev_", rel_a)

# RT stefanoleone992 reviews -> sl_rotten_tomatoes_link
rev_s = pd.read_csv(RAW / "stefanoleone992/rotten_tomatoes_critic_reviews.csv", low_memory=False,
                    usecols=["rotten_tomatoes_link", "critic_name", "publisher_name", "review_type", "top_critic",
                             "review_date", "review_score", "review_content"])
n0 = len(rev_s)
rev_s = rev_s.drop_duplicates().drop(columns="review_content")   # identical in every column incl. text (Edwards-A10 §2b); text not kept
print(f"RT stefanoleone992 reviews: dropped {n0 - len(rev_s):,} exact duplicate rows")
rel_s = (films.loc[films["sl_rotten_tomatoes_link"].notna(), ["sl_rotten_tomatoes_link", "release_dt"]]
              .rename(columns={"sl_rotten_tomatoes_link": "key"}))
agg_s = rollup(rev_s, "rotten_tomatoes_link", rev_s["review_type"].eq("Fresh"), rev_s["top_critic"],
               "review_date", "slrev_", rel_s)

films = (films.drop(columns=[c for c in films.columns if c.startswith(("rtrev_", "slrev_"))])
              .merge(agg_a, left_on="rt_id", right_index=True, how="left")
              .merge(agg_s, left_on="sl_rotten_tomatoes_link", right_index=True, how="left"))
assert films["id"].is_unique

u = films[films["usable"]]
pd.DataFrame({
    "usable films with reviews": [u["rtrev_n_reviews"].notna().mean(), u["slrev_n_reviews"].notna().mean()],
    "median reviews per film": [u["rtrev_n_reviews"].median(), u["slrev_n_reviews"].median()],
    "films with any review before release": [(u["rtrev_n_before_release"] > 0).mean(), (u["slrev_n_before_release"] > 0).mean()],
}, index=["RT andrezaza (rtrev_)", "RT stefanoleone992 (slrev_)"]).style.format(
    {"usable films with reviews": "{:.0%}", "median reviews per film": "{:.0f}",
     "films with any review before release": "{:.0%}"})

## 8. Hand-check sample

50 random The Numbers ↔ TMDB matches, weighted toward the riskier ones (year differs or budgets disagree). Review in `data/clean/edwards_tn_handcheck.csv`; mark `correct` as Y/N.

In [ ]:
rng = np.random.default_rng(560)
chk = tn_match.merge(tmdb[["id", "title", "original_title", "year", "runtime", "budget"]], on="id")
chk = chk.join(tn[["Movie Name", "year", "Running Time (minutes)", "Production Budget (USD)"]]
               .add_prefix("tn_"), on="src_row")
risky = chk[chk["year_diff"].gt(0) | ~chk["budget_agrees"]]
sample = pd.concat([risky.sample(25, random_state=rng.integers(1e9)),
                    chk.drop(risky.index).sample(25, random_state=rng.integers(1e9))])
sample["risky"] = sample.index.isin(risky.index)
sample["correct"] = ""
out = CLEAN / "edwards_tn_handcheck.csv"
if not out.exists():                      # written once, never overwritten (same rule as Matt's hand-check)
    sample.drop(columns="src_row").to_csv(out, index=False)
print(f"{len(risky):,} risky matches of {len(chk):,}; sample at {out}")

### 8b. Hand-check result

Reviewed by Tanner, 2026-11-22.

In [ ]:
from scipy.stats import beta

hc = pd.read_csv(CLEAN / "edwards_tn_handcheck.csv")
hc["ok"] = hc["correct"].astype(str).str.strip().str.upper().str.startswith("Y")
assert hc["correct"].notna().all(), "hand-check not finished"

def upper95(wrong, n):          # Clopper-Pearson exact 95% upper bound
    return beta.ppf(0.975, wrong + 1, n - wrong) if wrong < n else 1.0

res = hc.groupby("risky")["ok"].agg(checked="size", correct="sum")
res.loc["all"] = res.sum()
res["wrong"] = res["checked"] - res["correct"]
res["error rate 95% CI upper"] = [f"{upper95(w, n):.1%}" for w, n in zip(res["wrong"], res["checked"])]
res

## 9. One row per film

The sources are joined one-to-one, but TMDB itself sometimes lists the same film twice (and junk entries repeat). Rows are merged when they are the same film by either rule:

1. **Same IMDb id** (`imdb_id`), or
2. **Same normalized title, release year within ±1, runtime within 5 minutes (or unknown on either side: TMDB uses 0 for unknown), and combined budget within 15%.** Title alone isn't enough: remakes and shorts share titles (*Trolls* 2016 at $125M vs a $30K short).

Within a group, the row with the most filled-in values is kept (TMDB rows and usable rows first), and its blanks are filled from the others. `merged_ids` lists the absorbed ids. `usable` is recomputed after merging.

Then each film gets **`film_uid`**: `tmdb-<id>`, or `tn-<The Numbers URL slug>` (e.g. `tn-Marvels-The-(2023)`).

In [ ]:
unified = films[films["year"].between(*YEARS) & (films["fin_budget"].notna() | films["fin_revenue"].notna())].copy()
unified = unified.reset_index(drop=True)

# Candidate pairs (row positions)
pairs = []
imdb = unified["imdb_id"].where(unified["imdb_id"].astype(str).str.startswith("tt"))
g = unified.assign(imdb=imdb).dropna(subset=["imdb"]).groupby("imdb").groups
pairs += [(ix[0], j) for ix in g.values() if len(ix) > 1 for j in ix[1:]]

k = pd.DataFrame({"pos": unified.index, "key": norm_title(unified["title"]), "year": unified["year"],
                  "rt": unified["runtime"].replace(0, np.nan), "b": unified["fin_budget"]})
k = k[k["key"].ne("")]
pp = k.merge(k, on="key", suffixes=("", "_2"))
runtime_ok = (pp["rt"] - pp["rt_2"]).abs().le(5) | pp["rt"].isna() | pp["rt_2"].isna()   # 0/missing = unknown
pp = pp[(pp["pos"] < pp["pos_2"]) & (pp["year"] - pp["year_2"]).abs().le(1) & runtime_ok
        & ((pp["b"] / pp["b_2"] - 1).abs() < 0.15)]
pairs += list(zip(pp["pos"], pp["pos_2"]))

# Group pairs (union-find)
parent = {}
def find(x):
    while parent.get(x, x) != x:
        x = parent[x]
    return x
for a, b in pairs:
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)
groups = pd.Series({x: find(x) for x in set(np.ravel(pairs))}) if pairs else pd.Series(dtype=int)

unified["merged_ids"] = pd.NA
drop = []
filled = unified.notna().sum(axis=1)
for root, members in groups.groupby(groups).groups.items():
    rows = list(members)
    order = sorted(rows, key=lambda r: (unified.at[r, "in_tmdb"], unified.at[r, "usable"], filled[r]), reverse=True)
    keep, rest = order[0], order[1:]
    for r in rest:
        unified.loc[keep] = unified.loc[keep].combine_first(unified.loc[r])
    unified.at[keep, "merged_ids"] = ",".join(str(int(unified.at[r, "id"])) for r in rest)
    drop += rest

before_usable = unified["usable"].sum()
unified = unified.drop(index=drop).reset_index(drop=True)
unified["usable"] = (unified["year"].between(*YEARS) & unified["fin_budget"].ge(BUDGET_FLOOR)
                     & unified["fin_revenue"].gt(0))
assert unified["id"].is_unique
print(f"Merged {len(drop):,} duplicate rows into {groups.nunique() if len(groups) else 0:,} films "
      f"({len(pairs) - len(pp):,} pairs by IMDb id, {len(pp):,} by title/year/runtime/budget). "
      f"Usable: {before_usable:,} -> {unified['usable'].sum():,}")
display(unified.loc[unified["merged_ids"].notna() & unified["usable"], ["id", "title", "year", "fin_budget", "merged_ids"]])

# Stable key: TMDB id, or The Numbers URL slug for films not in TMDB (negative ids change between rebuilds)
slug = unified["tn_Movie URL"].str.extract(r"/movie/([^#?]+)")[0]
unified.insert(0, "film_uid", np.where(unified["in_tmdb"], "tmdb-" + unified["id"].astype("Int64").astype(str), "tn-" + slug))
assert unified["film_uid"].notna().all() and unified["film_uid"].is_unique


## 9b. Reviews table (one row per review)

A film has many reviews, so the reviews get their own table, `reviews_unified.csv`, linked to films by `film_uid`. Both review files are stacked with a `source` column. The films table keeps the per-film summaries (`rtrev_*`, `slrev_*`).

| Column | Meaning |
|---|---|
| `film_uid` | Film key (joins to `films_unified.csv`) |
| `source` | `rt_andrezaza` or `rt_stefanoleone992` |
| `review_id` | andrezaza `reviewId` (stefanoleone992 has none) |
| `review_date`, `before_release` | Review date; True if dated before the film's TMDB release date |
| `critic`, `publication`, `top_critic` | Who wrote it |
| `fresh` | RT fresh (True) / rotten (False) |
| `original_score` | The critic's own score as written ("3/5", "B+", …), often blank |

The two sources overlap (the same review can appear in both); they're kept apart by `source`, not merged. Review text isn't copied. ⚠️ Reviews are post-release: descriptive only, except possibly `before_release` reviews (plan step 11).

In [ ]:
xa = unified.loc[unified["rt_id"].notna(), ["film_uid", "rt_id", "release_dt"]]
ra = rev_a.merge(xa, left_on="id", right_on="rt_id")
ra = pd.DataFrame({"film_uid": ra["film_uid"], "source": "rt_andrezaza", "review_id": ra["reviewId"],
                   "review_date": pd.to_datetime(ra["creationDate"], errors="coerce"), "release_dt": ra["release_dt"],
                   "critic": ra["criticName"], "publication": ra["publicatioName"], "top_critic": ra["isTopCritic"].astype(bool),
                   "fresh": ra["reviewState"].eq("fresh"), "original_score": ra["originalScore"]})

xs = unified.loc[unified["sl_rotten_tomatoes_link"].notna(), ["film_uid", "sl_rotten_tomatoes_link", "release_dt"]]
rs = rev_s.merge(xs, left_on="rotten_tomatoes_link", right_on="sl_rotten_tomatoes_link")
rs = pd.DataFrame({"film_uid": rs["film_uid"], "source": "rt_stefanoleone992", "review_id": pd.NA,
                   "review_date": pd.to_datetime(rs["review_date"], errors="coerce"), "release_dt": rs["release_dt"],
                   "critic": rs["critic_name"], "publication": rs["publisher_name"], "top_critic": rs["top_critic"].astype(bool),
                   "fresh": rs["review_type"].eq("Fresh"), "original_score": rs["review_score"]})

reviews = pd.concat([ra, rs], ignore_index=True)
reviews["before_release"] = reviews["review_date"] < pd.to_datetime(reviews["release_dt"])
reviews = reviews.drop(columns="release_dt")
assert reviews["film_uid"].isin(unified["film_uid"]).all()
reviews.to_csv(CLEAN / "reviews_unified.csv", index=False)
print(f"reviews_unified.csv: {len(reviews):,} reviews of {reviews['film_uid'].nunique():,} films")
reviews.groupby("source").agg(reviews=("film_uid", "size"), films=("film_uid", "nunique"),
                              pct_fresh=("fresh", "mean"), before_release=("before_release", "sum"))

## 10. Save the unified dataset

In [ ]:
# One row per film: 2000-2023 films with at least one budget or revenue value from any source
# (TMDB films, plus The Numbers-only films with negative ids). Fixed file name: every other
# notebook reads this instead of re-merging. Gitignored; rebuilt on every run.
assert unified["id"].is_unique
unified.to_csv(CLEAN / "films_unified.csv", index=False)
print(f"films_unified.csv: {len(unified):,} films x {unified.shape[1]} columns "
      f"({unified['usable'].sum():,} usable, {(~unified['in_tmdb']).sum():,} not in TMDB)")